# 분석 개요

3. 시간권 첫구매유저의 웹/앱 행동
4. 1일권 첫구매유저의 웹/앱 행동
5. 기간권(무제한패스) 첫구매유저의 웹/앱 행동
6. 기간권(주말/야간패스) 첫구매유저의 웹/앱 행동

In [ ]:

# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pem_path,
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:

    # 3. DB 연결 (로컬 포트를 통해)
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_payment = pd.read_sql("""
                        SELECT

                        TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS p_date,
                        c.contract_uid transaction_id,
                        c.client_uid uid,
                        c.status,
                        c.product_name,
                        pp.name product_period,
                        TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
                        TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
                        TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date,
                        c.actual_price,
                        TO_CHAR(c.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS created_at,
                        c.is_migrated,
                        p.order_id,
                        u.phone_number
                        FROM contract c
                        LEFT JOIN payment p
                        ON c.contract_uid = p.contract_payment_uid
                        LEFT JOIN price_policy pp
                        ON c.price_policy_uid = pp.price_policy_uid
                        LEFT JOIN client u
                        ON c.client_uid = u.client_uid
                        LEFT JOIN payment_history ph
                        ON contract_uid = ph.payment_uid
                        WHERE status NOT IN ('WITHDRAW', 'CANCELED')
                        AND c.actual_price > 0
                        AND ph.payment_status = 'DONE'

                        ORDER BY ph.requested_at ASC;
                    """, conn)
    conn.close()
df_payment['product_name'] = df_payment['product_name'] + ' ' + df_payment['product_period'].astype(str)
df_payment = df_payment.drop(columns=['product_period'])
df_payment = df_payment[df_payment['p_date'] <= '2025-12-31']
df_payment.tail()

#구매데이터 전처리

In [ ]:
df_payment['p_date'] = pd.to_datetime(df_payment['p_date'])
df_payment = df_payment.sort_values(by=['phone_number', 'p_date'], ascending=True)
df_payment['row_num'] = df_payment.groupby('phone_number').cumcount() + 1

def classify_purchase(row):
    if row['row_num'] > 1:
        return '재구매'
    else:
        return '첫구매'

df_payment['purchase_type'] = df_payment.apply(classify_purchase, axis=1)
df_payment.head()

In [ ]:
df_payment_유니버스 = df_payment[df_payment['p_date'] >= '2025-06-11']
df_payment_유니버스.head()

In [ ]:
df_payment_유니버스['product_name'].unique()

In [ ]:
df_첫구매유저 = df_payment_유니버스[df_payment_유니버스['purchase_type'] == '첫구매'].sort_values('p_date', ascending = True)
df_첫구매유저.head()

In [ ]:
df_첫구매유저.info()

In [ ]:
# 1. 남길 컬럼 리스트 정의
cols_to_keep = ['p_date', 'uid', 'product_name', 'actual_price', 'created_at']

# 2. 필요한 컬럼만 선택하여 데이터프레임 업데이트
df_첫구매유저 = df_첫구매유저[cols_to_keep]

# 3. 결과 확인
df_첫구매유저.head()

In [ ]:
df_첫구매유저['product_name'].unique()

In [ ]:
import numpy as np

# 1. 분류 조건 설정 (순서대로 검사하므로 중요도가 높은 순으로 배치하세요)
conditions = [
    df_첫구매유저['product_name'].str.contains('시간', na=False),
    df_첫구매유저['product_name'].str.contains('1일', na=False),
    df_첫구매유저['product_name'].str.contains('주말', na=False)
]

# 2. 각 조건에 매칭되는 카테고리명
choices = ['시간패스', '1일권', '주말&야간패스']

# 3. 새로운 컬럼 생성 (어떤 조건에도 해당하지 않으면 '무제한패스')
df_첫구매유저['카테고리'] = np.select(conditions, choices, default='무제한패스')
df_첫구매유저.head()

In [ ]:
df_첫구매유저.groupby('카테고리')['uid'].nunique().reset_index()

#페이지 전처리

In [ ]:

sql_query = """
            SELECT *
            FROM MKT.MKT_eventAll_include_page AS t1
            LEFT JOIN MKT.MKT_userId_match_web AS t2
            ON t1.user_pseudo_id = t2.user_pseudo_id

            """
df = client.query(sql_query).to_dataframe()
# event_timestamp를 기준으로 정렬한 후, user_id별로 순차적인 번호를 매김
df.head()

In [ ]:
df['매체'] = np.where(df['page'].str.contains('https://fivespot.io/', na=False), '웹', '앱')
# '매체' 열의 값이 '앱'이 아닌 행들만 선택하여 기존 데이터프레임을 갱신합니다.
df_web = df[df['매체'] != '앱']
df_web.head()

## 필수 페이지 제외

In [ ]:
# 삭제할 값들의 리스트를 생성합니다.
remove_values = ['MainActivity', 'UIViewController', 'RNSScreen', 'https://fivespot.io/login']

# 'login', 'sign', 'client' 중 하나라도 포함하는 행을 찾기 위한 정규 표현식 패턴
pattern = 'login|sign|client|order'

# 'page' 열에서 위 패턴을 포함하지 않는(~) 행만 선택하여 데이터프레이을 갱신합니다.
# na=False는 결측치(NaN)가 있는 경우 오류를 방지합니다.
df_web = df_web[~df_web['page'].str.contains(pattern, na=False)]



# 'page' 열의 값이 remove_values 리스트에 포함되지 않은 행만 선택하여
# 기존 데이터프레임에 다시 할당합니다.
df_web = df_web[~df_web['page'].isin(remove_values)]
df_web.head()

In [ ]:
import numpy as np

# 1. 데이터 타입 일치 및 공백 제거 (문자열로 통일하는 것이 가장 확실합니다)
df_web['user_id'] = df_web['user_id'].astype(str).str.strip()
df_첫구매유저['uid'] = df_첫구매유저['uid'].astype(str).str.strip()

# 2. 매핑 재수행
df_web['구매여부'] = np.where(df_web['user_id'].isin(df_첫구매유저['uid']), '구매유저', '미구매유저')

# 3. 결과 확인
print(df_web['구매여부'].value_counts()) # 구매유저/미구매유저 분포 확인
df_web.head()

In [ ]:
df_web['구매여부'].unique()

# 미구매유저 분석

In [ ]:
df_미구매유저 = df_web[df_web['구매여부'] == '미구매유저']
df_미구매유저.head()

In [ ]:
import numpy as np

# 1. 빈 문자열, 문자열 'None', 'nan' 등을 모두 실제 NaN으로 변경
df_미구매유저['user_id_2'] = df_미구매유저['user_id'].replace(['', 'None', 'nan', None], np.nan)

# 2. 그 후 fillna 수행
df_미구매유저['user_id_2'] = df_미구매유저['user_id_2'].fillna(df_미구매유저['user_pseudo_id'])

# 결과 확인
print(df_미구매유저[['user_id', 'user_pseudo_id', 'user_id_2']].head())

In [ ]:
df_미구매유저['user_id'].unique()

In [ ]:
df_미구매유저.info()

In [ ]:
# 1. user_id_2와 event_timestamp를 기준으로 오름차순 정렬
df_미구매유저 = df_미구매유저.sort_values(by=['user_id_2', 'event_timestamp'])

# 2. page 컬럼이 이전 행과 달라지는 지점 찾기 (유저별로 그룹화하여 수행)
# 이전 행과 값이 다르면 True(1), 같으면 False(0)가 반환됩니다.
df_미구매유저['page_changed'] = (
    df_미구매유저.groupby('user_id_2')['page']
    .shift() != df_미구매유저['page']
).astype(int)

# 3. 유저별로 변화 지점을 누적 합산(cumsum)하여 넘버링 생성
df_미구매유저['page_row_num'] = df_미구매유저.groupby('user_id_2')['page_changed'].cumsum()

# 임시로 생성한 컬럼 삭제 (필요 시 유지해도 됨)
df_미구매유저 = df_미구매유저.drop(columns=['page_changed'])

# 결과 확인
df_미구매유저.head()

In [ ]:
df_미구매유저.info()

In [ ]:
# 1. 시간순 정렬 (가장 위 행을 남기기 위해 필수)
df_미구매유저 = df_미구매유저.sort_values(by=['user_id_2', 'event_timestamp'])

# 2. user_id_2와 page_row_num이 같은 행들 중 첫 번째만 남기고 제거
# subset을 지정하면 다른 컬럼(t_name, row_num 등)이 달라도 중복으로 간주합니다.
df_미구매유저_unique = df_미구매유저.drop_duplicates(
    subset=['user_id_2', 'page_row_num'],
    keep='first'
)
df_미구매유저_unique.head()

In [ ]:
df_미구매유저_unique.info()

In [ ]:
df_미구매유저_unique = df_미구매유저_unique[df_미구매유저_unique['event_date'] <= '20251231']
df_미구매유저_unique.head()

In [ ]:
df_미구매유저_groupby = df_미구매유저_unique.groupby(['page'])['user_id_2'].nunique().reset_index().sort_values(['user_id_2'], ascending = False)
df_미구매유저_groupby.head()

In [ ]:
df_미구매유저_groupby_1st = df_미구매유저_unique[df_미구매유저_unique['page_row_num'] == 1].groupby(['page'])['user_id_2'].nunique().reset_index().sort_values(['user_id_2'], ascending = False)
df_미구매유저_groupby_1st.head()

In [ ]:
# 1. 유저(user_id_2)별로 page_row_num이 가장 큰 행의 인덱스를 추출
last_page_indices = df_미구매유저_unique.groupby('user_id_2')['page_row_num'].idxmax()

# 2. 해당 인덱스로 데이터 필터링 (유저별 마지막 방문 페이지만 남음)
df_last_pages = df_미구매유저_unique.loc[last_page_indices]

# 3. 마지막 페이지별로 유저 수 집계
df_미구매유저_groupby_last = (
    df_last_pages.groupby(['page'])['user_id_2']
    .nunique()
    .reset_index()
    .sort_values(['user_id_2'], ascending=False)
)

# 결과 확인
df_미구매유저_groupby_last.head()

In [ ]:
df_미구매유저_unique.info()

#구매유저 분석

In [ ]:
df_첫구매유저['uid'] = df_첫구매유저['uid'].astype(str)
df_purchase_첫구매 = pd.merge(df_첫구매유저, df_web, left_on='uid', right_on='user_id', how='left')
df_purchase_첫구매.head()

In [ ]:
df_purchase_첫구매.info()

In [ ]:
import pandas as pd
import numpy as np

# 1. 새로운 변수에 복사 및 인덱스 초기화
df_purchase_첫구매_2 = df_purchase_첫구매.copy().reset_index(drop=True)

# 2. [중요] created_at_dt 컬럼이 없는 경우를 대비해 생성 로직 포함
if 'created_at_dt' not in df_purchase_첫구매_2.columns:
    df_purchase_첫구매_2['created_at_dt'] = pd.to_datetime(df_purchase_첫구매_2['created_at'])
    # 이미 timezone이 설정되어 있는지 확인 후 설정
    if df_purchase_첫구매_2['created_at_dt'].dt.tz is None:
        df_purchase_첫구매_2['created_at_dt'] = df_purchase_첫구매_2['created_at_dt'].dt.tz_localize('Asia/Seoul')

# 3. event_timestamp 결측치 제거
df_purchase_첫구매_2 = df_purchase_첫구매_2.dropna(subset=['event_timestamp']).copy()

# 4. event_timestamp 변환 (Int64 버그 방지)
ts_values = df_purchase_첫구매_2['event_timestamp'].values.astype(float)
df_purchase_첫구매_2['event_time_dt'] = pd.to_datetime(ts_values, unit='us', errors='coerce')

# 5. 타임존 처리 (UTC -> KST)
df_purchase_첫구매_2 = df_purchase_첫구매_2.dropna(subset=['event_time_dt']).copy()
df_purchase_첫구매_2['event_time_dt'] = df_purchase_첫구매_2['event_time_dt'].dt.tz_localize('UTC')
df_purchase_첫구매_2['event_time_kst'] = df_purchase_첫구매_2['event_time_dt'].dt.tz_convert('Asia/Seoul')

# 6. 구매 시간 이전 로그만 남기기
# 에러 방지를 위해 컬럼 존재 여부 재확인 후 비교
df_purchase_첫구매_2 = df_purchase_첫구매_2[
    df_purchase_첫구매_2['event_time_kst'] <= df_purchase_첫구매_2['created_at_dt']
].copy()

# 7. [추가 요청] 특정 페이지 제거 및 원본 변수 재할당
# page 컬럼이 https://fivespot.io/pass 인 행 제외
df_purchase_첫구매 = df_purchase_첫구매_2[df_purchase_첫구매_2['page'] != 'https://fivespot.io/pass'].copy()
df_purchase_첫구매 = df_purchase_첫구매.reset_index(drop=True)

# 최종 결과 확인
print(f"최종 df_purchase_첫구매 행 수: {len(df_purchase_첫구매)}")
df_purchase_첫구매.head()

In [ ]:
# 1. 먼저 event_timestamp를 기준으로 오름차순 정렬 (동일 유저 내에서 시간 순서 보장)
# uid도 함께 정렬 조건에 넣으면 그룹별로 모여 있어 확인이 용이합니다.
df_purchase_첫구매 = df_purchase_첫구매.sort_values(by=['uid', 'event_timestamp'])

# 2. 유저별(uid)로 그룹을 묶어 페이지가 이전 행과 다를 때마다 카운트를 올립니다.
df_purchase_첫구매['row_num_page'] = df_purchase_첫구매.groupby('uid')['page'].transform(
    lambda x: (x != x.shift()).cumsum()
)

# 결과 확인
df_purchase_첫구매_2.head()

In [ ]:

# 2. 데이터를 user_id_2, row_num_page, event_timestamp 순으로 정렬
# (조합별로 시간이 가장 빠른 행이 위로 오게 함)
df_purchase_첫구매 = df_purchase_첫구매.sort_values(by=['uid', 'row_num_page', 'event_timestamp'])

# 3. drop_duplicates를 사용하여 중복 제거
# subset: 중복을 판단할 기준 컬럼 (조합)
# keep='first': 정렬된 상태에서 가장 첫 번째(가장 빠른 시간) 행만 유지
df_purchase_첫구매 = df_purchase_첫구매.drop_duplicates(subset=['uid', 'row_num_page'], keep='first')

# 4. (선택) 인덱스 초기화
df_purchase_첫구매 = df_purchase_첫구매.reset_index(drop=True)
df_purchase_첫구매.head()

In [ ]:
# 1. user_id별로 event_time_kst의 최댓값과 최솟값의 차이를 계산 (일 단위)
# transform을 사용해 유저별 계산 결과를 모든 행에 동일하게 할당합니다.
df_purchase_첫구매['convDuration'] = df_purchase_첫구매.groupby('user_id')['event_time_kst'].transform(lambda x: (x.max() - x.min()).days)

# 2. 결과 확인 (유저별로 일수가 잘 들어갔는지 상위 10개 행 출력)
display(df_purchase_첫구매[['user_id', 'event_time_kst', 'convDuration']].sort_values(by=['user_id', 'event_time_kst']).head(10))

# 3. 데이터 분포 요약 (보고용)
print("\n[전환 소요 일수 요약]")
print(df_purchase_첫구매['convDuration'].describe())

In [ ]:
result = df_purchase_첫구매.groupby(['카테고리','convDuration'])['user_id'].nunique().reset_index()
result.head()

In [ ]:
# 1. 유저별/카테고리별로 중복을 제거하여 유저당 하나의 convDuration만 남깁니다.
df_user_category_duration = df_purchase_첫구매[['user_id', '카테고리', 'convDuration']].drop_duplicates()

# 2. 카테고리별로 평균(mean)과 중앙값(median) 계산
category_conv_stats = df_user_category_duration.groupby('카테고리')['convDuration'].agg(['mean', 'median', 'count']).reset_index()

# 3. 평균 일수 기준 내림차순 정렬 (고민이 긴 카테고리부터)
category_conv_stats = category_conv_stats.sort_values(by='mean', ascending=False)

# 컬럼명 변경 (보고용)
category_conv_stats.columns = ['카테고리', '평균 소요 일수', '중앙값(일)', '유저 수']

display(category_conv_stats)

In [ ]:
df_purchase_첫구매_2[df_purchase_첫구매_2['page'] == 'https://fivespot.io/pass'].head()

In [ ]:
# 1. 'page' 값이 'https://fivespot.io/pass'와 일치하지 않는 행만 필터링
# 포함(contains)이 아닌 정확한 일치(==)를 피하기 위해 != 연산자를 사용합니다.
df_purchase_첫구매 = df_purchase_첫구매_2[df_purchase_첫구매_2['page'] != 'https://fivespot.io/pass'].copy()

# 2. 인덱스 재정렬 (필터링 후 빈 번호를 채워줌)
df_purchase_첫구매 = df_purchase_첫구매.reset_index(drop=True)

# 3. 결과 확인
removed_count = len(df_purchase_첫구매_2) - len(df_purchase_첫구매)
print(f"제거된 행 수: {removed_count}개")
print(f"최종 데이터 행 수: {len(df_purchase_첫구매)}")

In [ ]:
df_purchase_첫구매['조회페이지수'] = df_purchase_첫구매.groupby('uid')['row_num_page'].transform('max')
df_purchase_첫구매 = df_purchase_첫구매.sort_values(['uid', 'event_timestamp'])
df_purchase_첫구매['first_page'] = df_purchase_첫구매.groupby('uid')['page'].transform('first')
df_purchase_첫구매['last_page'] = df_purchase_첫구매.groupby('uid')['page'].transform('last')
df_purchase_첫구매.head()


In [ ]:
result = df_purchase_첫구매.groupby(['카테고리','page'])['uid'].nunique().reset_index().sort_values(['카테고리','uid'], ascending = False)

In [ ]:
df_purchase_첫구매.info()

In [ ]:
df_미구매유저_unique.head()

In [ ]:
daily_미구매 = df_미구매유저_unique.groupby(['event_date','page'])['user_id_2'].nunique().reset_index().sort_values(['event_date'])
daily_미구매['event_date'] = pd.to_datetime(daily_미구매['event_date'], format='%Y%m%d')
daily_미구매.head()

In [ ]:
df_purchase_첫구매.head()

In [ ]:
# row_num_page가 1인 데이터에 대해 그룹화 및 집계
daily_첫구매 = df_purchase_첫구매[df_purchase_첫구매['row_num_page'] == 1].groupby(
    ['p_date', '카테고리', 'last_page']
).agg({
    'actual_price': 'sum',     # 판매 금액 합계
    'uid': 'nunique'           # 유니크 유저 수 (구매수)
}).reset_index()

# 컬럼명 변경 (uid -> 구매수)
daily_첫구매 = daily_첫구매.rename(columns={'uid': '구매수'})

# p_date 기준으로 정렬
daily_첫구매 = daily_첫구매.sort_values(['p_date'])

# 결과 확인
daily_첫구매.head()

In [ ]:
import pandas as pd


# =========================================================================
# 1. 프로모션 캘린더 타임라인 매핑 함수 (보안 마스킹 처리)
# =========================================================================
def get_promotion_name(event_date):
    """엑셀 마케팅 캘린더 일정을 기반으로 날짜별 캠페인 코드/타입을 반환합니다.

    사내 보안 가이드에 따라 실제 프로모션 명칭은 익명화(Masking) 처리되었습니다.
    """
    if pd.isna(event_date):
        return "NO_DATE"

    # 문자열 타입인 경우 datetime 포맷으로 동적 캐스팅
    if isinstance(event_date, str):
        event_date = pd.to_datetime(event_date)

    # -------------------------------------------------------------------------
    # 일별 프로모션 타임라인 조건 분기 (As-Is 마케팅 명칭 -> To-Be 기술 명칭 매핑)
    # -------------------------------------------------------------------------
    if pd.Timestamp("2025-06-12") <= event_date <= pd.Timestamp("2025-06-22"):
        return "PROMOTION_TYPE_ALPHA"
    elif pd.Timestamp("2025-06-23") <= event_date <= pd.Timestamp("2025-06-29"):
        return "PROMOTION_TYPE_BETA"
    elif pd.Timestamp("2025-06-30") <= event_date <= pd.Timestamp("2025-07-06"):
        return "PROMOTION_TYPE_GAMMA"
    elif pd.Timestamp("2025-07-07") <= event_date <= pd.Timestamp("2025-07-13"):
        return "CAMPAIGN_WEEK_A"
    elif pd.Timestamp("2025-07-14") <= event_date <= pd.Timestamp("2025-07-21"):
        return "FLASH_SALE_01"
    elif pd.Timestamp("2025-07-22") <= event_date <= pd.Timestamp("2025-07-28"):
        return "SEASONAL_PROMOTION_SUMMER_A"
    elif pd.Timestamp("2025-07-29") <= event_date <= pd.Timestamp("2025-08-04"):
        return "GROWTH_PROJECT_01"
    elif pd.Timestamp("2025-08-05") <= event_date <= pd.Timestamp("2025-08-11"):
        return "PROMOTION_TYPE_DELTA"
    elif pd.Timestamp("2025-08-12") <= event_date <= pd.Timestamp("2025-08-18"):
        return "PROMOTION_TYPE_EPSILON"
    elif pd.Timestamp("2025-08-19") <= event_date <= pd.Timestamp("2025-08-25"):
        return "CAMPAIGN_WEEK_A"
    elif pd.Timestamp("2025-08-26") <= event_date <= pd.Timestamp("2025-09-01"):
        return "GROWTH_PROJECT_01"
    elif pd.Timestamp("2025-09-02") <= event_date <= pd.Timestamp("2025-09-08"):
        return "SEASONAL_PROMOTION_AUTUMN_A"
    elif pd.Timestamp("2025-09-09") <= event_date <= pd.Timestamp("2025-09-15"):
        return "CAMPAIGN_WEEK_B"
    elif pd.Timestamp("2025-09-16") <= event_date <= pd.Timestamp("2025-09-22"):
        return "COUNTDOWN_SALE_01"
    elif pd.Timestamp("2025-09-23") <= event_date <= pd.Timestamp("2025-09-29"):
        return "BRAND_INBOUND_PROMOTION"
    elif pd.Timestamp("2025-09-30") <= event_date <= pd.Timestamp("2025-10-02"):
        return "FLASH_SALE_02"
    elif pd.Timestamp("2025-10-03") <= event_date <= pd.Timestamp("2025-10-13"):
        return "ACQUISITION_CAMPAIGN_01"
    elif pd.Timestamp("2025-10-14") <= event_date <= pd.Timestamp("2025-10-20"):
        return "PROMOTION_TYPE_ZETA"
    elif pd.Timestamp("2025-10-21") <= event_date <= pd.Timestamp("2025-10-27"):
        return "PROMOTION_TYPE_ETA"
    elif pd.Timestamp("2025-10-28") <= event_date <= pd.Timestamp("2025-11-03"):
        return "RETENTION_SUPPORT_CAPI"
    elif pd.Timestamp("2025-11-04") <= event_date <= pd.Timestamp("2025-11-10"):
        return "CHALLENGE_PROMOTION_01"
    elif pd.Timestamp("2025-11-11") <= event_date <= pd.Timestamp("2025-11-17"):
        return "CAMPAIGN_WEEK_C"
    elif pd.Timestamp("2025-11-18") <= event_date <= pd.Timestamp("2025-12-01"):
        return "SEASONAL_PROMOTION_WINTER_A"
    elif pd.Timestamp("2025-12-02") <= event_date <= pd.Timestamp("2025-12-08"):
        return "ANNIVERSARY_PROMOTION"
    elif pd.Timestamp("2025-12-09") <= event_date <= pd.Timestamp("2025-12-15"):
        return "CAMPAIGN_WEEK_D"
    elif pd.Timestamp("2025-12-16") <= event_date <= pd.Timestamp("2025-12-31"):
        return "SEASONAL_PROMOTION_WINTER_B"
    else:
        return "STANDARD_NON_PROMOTION"


# =========================================================================
# 2. 첫구매 시계열 코호트 데이터프레임 매핑 적용
# =========================================================================
# 타깃 결제일자(p_date) 파서 에러 방지를 위해 명시적 datetime 변환
daily_첫구매["p_date"] = pd.to_datetime(daily_첫구매["p_date"])

# 벡터라이징 연산 대체용 엘리먼트 와이즈 브로드캐스팅 매핑 적용
daily_첫구매["프로모션"] = daily_첫구매["p_date"].apply(get_promotion_name)

# =========================================================================
# 3. 데이터 정합성 파이프라인 샘플 스캔 (Outputs 제거 대상 영역)
# =========================================================================
daily_첫구매.head()

In [ ]:
# 1. event_date를 datetime 형식으로 변환 (비교 연산을 위해 필수)
daily_미구매['event_date'] = pd.to_datetime(daily_미구매['event_date'])

# 2. get_promotion_name 함수를 적용하여 '프로모션' 컬럼 생성
daily_미구매['프로모션'] = daily_미구매['event_date'].apply(get_promotion_name)

# 3. 결과 확인
daily_미구매.head()